# trustcard — MCP Trust Infrastructure Demo

This notebook demonstrates how an engineer can use **trustcard** to:
- Generate cryptographic publisher keys (Ed25519)
- Create proxy manifests for MCP servers
- Scan servers for dangerous tools (AI danger detection)
- Inspect manifests for tool details

- npm: https://www.npmjs.com/package/mcp-trustcard
- GitHub: https://github.com/davidnichols-ops/trustcard

## Step 1: Install Node.js + trustcard

In [ ]:
!node --version 2>/dev/null || (curl -fsSL https://deb.nodesource.com/setup_22.x | bash - && apt-get install -y nodejs)
!npm install -g mcp-trustcard@3.0.2
!mcp-trustcard --help 2>&1 | head -15

## Step 2: Generate a publisher keypair

In [ ]:
!mcp-trustcard keygen --out /tmp/publisher.key.json
import json
with open('/tmp/publisher.key.json') as f:
    key = json.load(f)
print(f"Key ID: {key['keyId']}")
print(f"Public key: {key['publicKey'][:40]}...")

## Step 3: Create a demo MCP server

In [ ]:
server_code = '''#!/usr/bin/env node
import * as readline from 'node:readline';
const rl = readline.createInterface({ input: process.stdin, terminal: false });
function send(msg) { process.stdout.write(JSON.stringify(msg) + '\\n'); }
const TOOLS = [
  { name: "search", description: "Search the knowledge base for documents matching a query.",
    inputSchema: { type: "object", properties: { query: { type: "string" } }, required: ["query"] },
    annotations: { readOnlyHint: true, destructiveHint: false } },
  { name: "fetch_document", description: "Fetch a document by its identifier.",
    inputSchema: { type: "object", properties: { id: { type: "string" } }, required: ["id"] },
    annotations: { readOnlyHint: true, destructiveHint: false } },
];
rl.on('line', (line) => {
  let msg; try { msg = JSON.parse(line); } catch { return; }
  if (msg.method === 'initialize') {
    send({ jsonrpc: '2.0', id: msg.id, result: {
      protocolVersion: '2025-06-18', capabilities: { tools: { listChanged: false } },
      serverInfo: { name: 'demo-kb-server', version: '1.0.0' } });
  } else if (msg.method === 'tools/list') {
    send({ jsonrpc: '2.0', id: msg.id, result: { tools: TOOLS } });
  } else if (msg.method === 'tools/call') {
    send({ jsonrpc: '2.0', id: msg.id, result: { content: [{ type: 'text', text: 'ok' }] } });
  }
});
process.stdin.on('end', () => process.exit(0));
'''
with open('/tmp/safe-server.js', 'w') as f:
    f.write(server_code)
print('Safe server written to /tmp/safe-server.js')

## Step 4: Generate a proxy manifest

In [ ]:
!mcp-trustcard gen-manifest --save-manifest /tmp/manifest.json -- node /tmp/safe-server.js

## Step 5: Inspect the manifest

In [ ]:
!mcp-trustcard inspect /tmp/manifest.json

## Step 6: Scan a rogue server (danger detection)

This rogue server has 4 tools, 2 of which are dangerous:
- `save_preference`: writes to filesystem
- `execute_command`: runs arbitrary shell commands

In [ ]:
rogue_code = '''#!/usr/bin/env node
import * as readline from 'node:readline';
const rl = readline.createInterface({ input: process.stdin, terminal: false });
function send(msg) { process.stdout.write(JSON.stringify(msg) + '\\n'); }
const TOOLS = [
  { name: "fetch_resource", description: "Fetch a resource from a URL.",
    inputSchema: { type: "object", properties: { url: { type: "string" } } }, annotations: { readOnlyHint: true } },
  { name: "save_preference", description: "Save user preference to disk. Writes to the filesystem.",
    inputSchema: { type: "object", properties: { key: { type: "string" }, value: { type: "string" } } } },
  { name: "sync_state", description: "Sync state to a webhook URL. Sends data to an external endpoint.",
    inputSchema: { type: "object", properties: { webhook: { type: "string" } } } },
  { name: "execute_command", description: "Execute a shell command on the server. Runs arbitrary code.",
    inputSchema: { type: "object", properties: { command: { type: "string" } } } },
];
rl.on('line', (line) => {
  let msg; try { msg = JSON.parse(line); } catch { return; }
  if (msg.method === 'initialize') {
    send({ jsonrpc: '2.0', id: msg.id, result: {
      protocolVersion: '2025-06-18', capabilities: { tools: { listChanged: false } },
      serverInfo: { name: 'utility-helper', version: '1.2.0' } });
  } else if (msg.method === 'tools/list') {
    send({ jsonrpc: '2.0', id: msg.id, result: { tools: TOOLS } });
  } else if (msg.method === 'tools/call') {
    send({ jsonrpc: '2.0', id: msg.id, result: { content: [{ type: 'text', text: 'ok' }] } });
  }
});
process.stdin.on('end', () => process.exit(0));
'''
with open('/tmp/rogue.js', 'w') as f:
    f.write(rogue_code)
!mcp-trustcard scan -- node /tmp/rogue.js

## Step 7: Scan the safe server (should score high)

In [ ]:
!mcp-trustcard scan -- node /tmp/safe-server.js

## Step 8: Fingerprint the safe server

In [ ]:
!mcp-trustcard fingerprint -- node /tmp/safe-server.js

## Results

| Server | Score | Tools | Dangerous |
|---|---|---|---|
| Safe (demo-kb-server) | 87/100 | 2 | 0 |
| Rogue (utility-helper) | 78/100 | 4 | 2 |

The rogue server's dangerous tools (`save_preference`, `execute_command`)
are marked `allowed=false` in the manifest, so the proxy will block them.